# 応用編3
このノートブックでは、共通コードをライブラリ化する例を示します。  
ここでは、ドメインの初期化コードを利用する方法を用います。

## 概要
ドメイン初期化コードは、スマートコントラクトの実行前に行われる初期化処理を定義したものです。  
ドメイン初期化コードとして共通的に利用するファンクションを定義しておくことにより、  
ドメイン内の複数のコントラクトから、これをライブラリのように利用することができます。  


## 準備

実行の準備を行います。

In [1]:
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract, createObjectF } = await import('../lib/notebook-util.mjs');

ドメイン初期化コードをバックアップします。

In [2]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_domain', id: domain });
var backup_code = resp.value.code;

キーバリューコントラクトを作成します。

In [3]:
var cid_kvs = await createObjectF('keyvalue', 'adv3_kvs');
console.log(cid_kvs);

c238294294


## ドメイン初期化コードの設定

共通コードとして、キーバリューコントラクトにアクセスするファンクションを定義します。  
(サンプルとしての処理であり、処理の内容に深い意味はありません）

In [4]:
function common_func_transfer(from, to){
    var cnt = openContract('adv3_kvs');
    var fval = +cnt.get(from);
    var tval = +cnt.get(to);
    if(fval < 0) throw new Error('fval<0');
    if(tval > 0) throw new Error('tval>0');
    fval--;
    tval++;
    cnt.set(from, fval);
    cnt.set(to, tval);
    return [from, fval, to, tval];
}

上記のファンクションをドメイン初期化コードとして設定します。

In [5]:
var code = common_func_transfer.toString();
var resp = await rpc.call(adminWallet, 'c1update', { id: '@'+domain, prop: 'code', value: code });
console.log(resp);

{
  txno: 701967,
  txid: 'xC9XUDWmBatZ6TkoVRRVVQEFwHPcLgwb2ZJ5cLmAWr6nAB',
  status: 'ok',
  value: null
}


## スマートコントラクトのデプロイ

共通コードを利用するスマートコントラクトを2つデプロイします。  
両方のコントラクトから同じファンクションcommon_func_transferを利用します。

In [6]:
var cid1 = await deploySmartContract(function adv3_1(){
    // do something
    return common_func_transfer(getCallerId(), getContractId());
});
console.log(cid1);

c020334299


In [7]:
var cid2 = await deploySmartContract(function adv3_2(){
    // do something
    return common_func_transfer(getContractId(), getCallerId());
});
console.log(cid2);

c004924940


デプロイしたスマートコントラクトがキーバリューにアクセスできるように許可します。

In [8]:
await rpc.call(adminWallet, 'c1update', { id: cid_kvs, prop: 'add accessible_to', value: [cid1, cid2] });

{
  txno: 701974,
  txid: 'xyd7E5vgHTk5b2edEY2VJamxWzHaPkgpoq432VrNGbR2NB',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

cid1を実行します。

In [9]:
var resp = await rpc.call(adminWallet, cid1);
console.log(resp);

{
  txno: 701975,
  txid: 'xNNVU8J6qxRn2mB42zQTJCC58D9TqZ4tpGGb2p878aSiPB',
  status: 'ok',
  value: [ 'u58281888', -1, 'c020334299', 1 ]
}


cid2を実行します。

In [10]:
var resp = await rpc.call(adminWallet, cid2);
console.log(resp);

{
  txno: 701980,
  txid: 'x7gKBUbAMDEXfoVFDUwVt34WxGY27KeF7P4YnzYWxMkg8',
  status: 'ok',
  value: [ 'c004924940', -1, 'u58281888', 0 ]
}


## エラーになる場合

ここでcid2を実行するとエラーになります。  
（このエラーはサンプルのために起こしたものであり、深い意味はありません）  
スタックトレースから、共通コードの中で例外がスローされたことがわかります。

In [11]:
var resp = await rpc.call(adminWallet, cid2);
console.log(resp);

{
  txno: 701985,
  txid: 'xJKUVHbh2MsuPSdTTAFiMwDQqf2XdcmSHsgc8SjfoUxmr',
  status: 'aborted',
  value: 'Error: fval<0\n' +
    '    at common_func_transfer (d93319890:1:161)\n' +
    '    at c004924940:1:16'
}


## クリーンアップ
このノートブックで設定したドメイン初期化コードは本来のものではないので、元に戻しておきます。  

In [12]:
var resp = await rpc.call(adminWallet, 'c1update', { id: '@'+domain, prop: 'code', value: backup_code });
console.log(resp);

{
  txno: 701988,
  txid: 'xKX2KZec3mjcZLH8QnqKvsSsUuwpPFijcRn24rkdcWKD5',
  status: 'ok',
  value: null
}
